In [1]:
%pip install nflreadpy


Note: you may need to restart the kernel to use updated packages.


In [2]:
import numpy as np
import nflreadpy as nfl
import pandas as pd

In [3]:
TRAIN_SEASONS = list(range(2021, 2025))  # seasons used to build the model, exclude 2025 for coaching comparison
EVAL_SEASON = 2025                        # season to grade real decisions on
MAX_YTG = 15                              # 15+ yards to go are pooled together
MIN_TIME_TO_SCORE = 30                    # seconds you need after getting the ball back
END_OF_HALF_SECS = 5                      # at or under this, it's the last play of the half
TD_POINTS = 7.0
FG_POINTS = 3.0
  
COLS = [
    "season", "week", "game_id", "posteam", "defteam", "qtr", "down", "ydstogo",
    "yardline_100", "play_type", "ep", "epa", "yards_gained", "fumble_lost",
    "interception", "penalty", "qb_kneel", "qb_spike", "kick_distance",
    "field_goal_result", "game_seconds_remaining", "score_differential",
    "posteam_timeouts_remaining", "desc",
]
 
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 80)
 
pbp = nfl.load_pbp(seasons=TRAIN_SEASONS).select(COLS).to_pandas()
pbp_eval = nfl.load_pbp(seasons=EVAL_SEASON).select(COLS).to_pandas()  # load evaluation season separately
clean = (pbp["qb_kneel"].fillna(0) == 0) & (pbp["qb_spike"].fillna(0) == 0)
clean_eval = (pbp_eval["qb_kneel"].fillna(0) == 0) & (pbp_eval["qb_spike"].fillna(0) == 0)

In [4]:
# 1. Build the lookup tables from the data
 
# EP of having 1st down at each yard line (index = yardline_100) 
first = pbp[(pbp["down"] == 1) & pbp["ep"].notna() & pbp["yardline_100"].between(1, 99)]
ep1 = (
    first.groupby(first["yardline_100"].astype(int))["ep"].mean()
    .reindex(range(1, 100))
    .interpolate(limit_direction="both")
    .rolling(5, center=True, min_periods=1).mean()
)
EP1 = np.zeros(101)
EP1[1:100] = ep1.to_numpy()

In [5]:
# Conversion rate by yards to go (3rd + 4th down run/pass plays) 
att = pbp[
    pbp["down"].isin([3, 4])
    & pbp["play_type"].isin(["run", "pass"])
    & clean
    & (pbp["penalty"].fillna(0) == 0)
    & pbp["ydstogo"].notna()
    & pbp["yards_gained"].notna()
].copy()
att["ytg"] = att["ydstogo"].astype(int).clip(1, MAX_YTG)
att["converted"] = (
    (att["yards_gained"] >= att["ydstogo"])
    & (att["fumble_lost"].fillna(0) == 0)
    & (att["interception"].fillna(0) == 0)
)
conv = (
    att.groupby("ytg")["converted"].mean()
    .reindex(range(1, MAX_YTG + 1))
    .interpolate(limit_direction="both")
)
P_CONVERT = np.zeros(MAX_YTG + 1)
P_CONVERT[1:] = np.minimum.accumulate(conv.to_numpy())

In [6]:
# EP after a successful conversion, using real yards gained on conversions
SUCCESS = np.full((100, MAX_YTG + 1), np.nan)
TD_SHARE = np.zeros((100, MAX_YTG + 1))  # share of conversions that go for a TD
for k in range(1, MAX_YTG + 1):
    gains = att.loc[att["converted"] & (att["ytg"] == k), "yards_gained"].astype(int).to_numpy()
    for y in range(k, 100):
        new_spot = y - gains
        SUCCESS[y, k] = np.where(
            new_spot <= 0, TD_POINTS, EP1[np.clip(new_spot, 1, 99)]
        ).mean()
        TD_SHARE[y, k] = (new_spot <= 0).mean()

In [7]:
# Punt value by yard line: average EP after the punt (punting team's view) 
punts = pbp[(pbp["play_type"] == "punt") & pbp["ep"].notna() & pbp["epa"].notna()].copy()
punts["y"] = punts["yardline_100"].astype(int)
punts["ep_after"] = punts["ep"] + punts["epa"]
pg = punts.groupby("y")["ep_after"].agg(["mean", "count"]).reindex(range(1, 100))
punt_ok = pg["count"].fillna(0) >= 20  # only where teams actually punt
punt_ep = (
    pg["mean"].where(punt_ok)
    .rolling(5, center=True, min_periods=1).mean()
    .where(punt_ok)
)
PUNT = np.full(100, np.nan)
PUNT[1:] = punt_ep.to_numpy()

# calculate punt means per field position to gather disappointment value adjustments
punts_mean = punts.groupby("y")["ep_after"].transform("mean") # group by field position (y) and get mean of EP
# compare real punt result EP with mean 
punts["disappointment_punt"] = (punts_mean - punts["ep_after"]).clip(lower = 0) # non-negative    
# avg disappointment value by field position
punt_dp_avg = (punts.groupby("y")["disappointment_punt"]
               .mean()
               .reindex(range(100))
               .to_numpy()
)

In [8]:
# Field goal make rate by kick distance 
fgs = pbp[(pbp["play_type"] == "field_goal") & pbp["kick_distance"].notna()].copy()
fgs["dist"] = fgs["kick_distance"].astype(int)
fgs["made"] = (fgs["field_goal_result"] == "made").astype(int)
fd = fgs.groupby("dist")["made"].agg(["sum", "count"]).reindex(range(18, 80), fill_value=0)
made_roll = fd["sum"].rolling(5, center=True, min_periods=1).sum()
att_roll = fd["count"].rolling(5, center=True, min_periods=1).sum()
fg_rate = (made_roll / att_roll).where(att_roll >= 10).bfill().fillna(0).cummin()
 
yards = np.arange(100)
P_FG = np.array([fg_rate.get(y + 17, 0.0) for y in yards])  # snap 7 + end zone 10
# Missed FG: opponent takes over at the spot of the kick, or their 20 if inside it
miss_opp = np.clip(np.where(yards + 7 <= 20, 80, 100 - (yards + 7)), 1, 99)
FG_EP = np.where(P_FG > 0, P_FG * FG_POINTS + (1 - P_FG) * -EP1[miss_opp], np.nan)

In [9]:
def evaluate_disappointment(p, g, b): # find modifier value for disappointment to adjust from Amer's EP values
    # p = probability of good result (conversion, successful field goal, good punt*)
    # g = estimated EP of good result (converting first down, successful field goal, good punt*)
    # b = estimated EP of bad result (turnover on downs, missed field goal, bad punt*)
    disappointment_measure = p * (1 - p) * (g - b)
    return disappointment_measure

In [10]:
def evaluate_frame(df: pd.DataFrame, d = 0.6) -> pd.DataFrame: 
    """Evaluate every row. Needs columns: qtr, game_seconds_remaining,
    yardline_100, ydstogo, score_differential, posteam_timeouts_remaining."""
    y = df["yardline_100"].astype(int).clip(1, 99).to_numpy()
    ytg = np.minimum(df["ydstogo"].astype(int).clip(lower=1).to_numpy(), y)
    k = np.minimum(ytg, MAX_YTG)
 
    p = P_CONVERT[k]
    succ = np.where(ytg >= y, TD_POINTS, SUCCESS[y, k])  # 4th & goal: converting = TD
    fail = -EP1[100 - y]                                  # opponent gets it here
    go = p * succ + (1 - p) * fail
    punt = PUNT[y]
    disappointment_punt = punt_dp_avg[y]
    fg = FG_EP[y]
    secs = df["game_seconds_remaining"].to_numpy(dtype=float)
    qtr = df["qtr"].to_numpy()
 
    # Situation override 1: last play of the half / game
    # Nothing after this play counts, so only points scored ON the play matter:
    # a first down short of the end zone is worth 0, and so is giving the ball away.
    half_secs = np.where(qtr == 2, secs - 1800, np.where(qtr >= 4, secs, np.inf))
    last_play = half_secs <= END_OF_HALF_SECS
    td_share = np.where(ytg >= y, 1.0, TD_SHARE[y, k])
    succ = np.where(last_play, td_share * TD_POINTS, succ)
    fail = np.where(last_play, 0.0, fail)
    go = p * succ + (1 - p) * fail
    disappointment_convert = evaluate_disappointment(p, succ, fail) # disappointment adjustment of failure in going for a conversion
    punt = np.where(last_play & ~np.isnan(punt), 0.0, punt)
    fg = np.where(last_play & ~np.isnan(fg), P_FG[y] * FG_POINTS, fg)
 
    # --- Situation override 2: trailing late, can you get the ball back? ---
    tos = df["posteam_timeouts_remaining"].fillna(3).clip(0, 3).to_numpy()
    stops = tos + (secs > 120)                      # two-minute warning acts as a stop
    kneel_out = 40 * np.clip(3 - stops, 0, 3) + 10  # time opponent can burn kneeling
    behind = -df["score_differential"].fillna(0).to_numpy(dtype=float)
    desperate = (
        (qtr >= 4)
        & (behind > 0)
        & (secs <= kneel_out + MIN_TIME_TO_SCORE)
    )
    punt = np.where(desperate & ~np.isnan(punt), fail, punt)              # punting ends it
    fg = np.where(desperate & (behind > 3) & ~np.isnan(fg), fail, fg)     # FG can't tie/win
    
    # disappointment for field goals --- situationally adjusted for Override 1 only ---
    fg_convert_p = P_FG[y]
    fg_succ = 3.0
    fg_fail = -EP1[miss_opp[y]]
    fg_fail =  np.where(last_play, 0.0, fg_fail)       # account for Situation override 1: last play of half / game
    # calculate disappointment measure for field goal good/bad outcomes
    disappointment_fg = evaluate_disappointment(fg_convert_p, fg_succ, fg_fail)

    # disappointment adjustment coefficient 
    # set between 0, 1 with the ability to move it up and down like a slider
    # EP(i) - (d * D(i))
    uncertainty_adj_go = go - (d * disappointment_convert)
    uncertainty_adj_fg = fg - (d * disappointment_fg)
    uncertainty_adj_punt = punt - (d * disappointment_punt)
    
    best_alt = np.fmax(punt, fg)
    best_alt = np.where(np.isnan(best_alt), fail, best_alt)
    options = np.column_stack([uncertainty_adj_go, uncertainty_adj_punt, uncertainty_adj_fg])
    rec = np.array(["go", "punt", "field_goal"])[np.nanargmax(options, axis=1)]
 
    return pd.DataFrame(
        {
            "p_convert": p,
            "ep_if_convert": succ,
            "ep_if_fail": fail,
            "ep_go": go,
            "ep_punt": punt,
            "ep_fg": fg,
            "best_alt_ep": best_alt,
            "points_if_convert": succ - best_alt,
            "points_if_fail": fail - best_alt,
            "breakeven_rate": np.clip((best_alt - fail) / (succ - fail), 0, 1),
            "go_advantage": go - best_alt,
            "recommendation": rec,
            "desperation": desperate,
            "last_play": last_play,
        },
        index=df.index,
    )

In [11]:
def evaluate(qtr, clock, yardline_100, ydstogo, score_diff, timeouts=3):
    """One situation. clock = time left in the quarter as "mm:ss".
    yardline_100 = yards from the opponent's end zone (1 = their 1, 99 = your 1).
    score_diff = your score minus theirs (negative = trailing)."""
    m, s = clock.split(":")
    secs_in_qtr = int(m) * 60 + int(s)
    game_secs = (4 - qtr) * 900 + secs_in_qtr if qtr <= 4 else secs_in_qtr
    row = pd.DataFrame([{
        "qtr": qtr,
        "game_seconds_remaining": game_secs,
        "yardline_100": yardline_100,
        "ydstogo": ydstogo,
        "score_differential": score_diff,
        "posteam_timeouts_remaining": timeouts,
    }])
    r = evaluate_frame(row).iloc[0]
 
    spot = (f"opp {yardline_100}" if yardline_100 < 50
            else "midfield" if yardline_100 == 50 else f"own {100 - yardline_100}")
    score = "tied" if score_diff == 0 else (f"up {score_diff}" if score_diff > 0 else f"down {-score_diff}")
    fmt = lambda v: "n/a" if pd.isna(v) else f"{v:+.2f}"
    q = "OT" if qtr == 5 else f"Q{qtr}"
 
    print(f"\n{q} {clock} | 4th & {ydstogo} at {spot} | {score} | {timeouts} TO")
    print(f"  P(convert):                {r.p_convert:.0%}")
    print(f"  Points added if you make:  {fmt(r.points_if_convert)}  (vs best kick/punt)")
    print(f"  Points lost if you miss:   {fmt(r.points_if_fail)}")
    print(f"  Break-even conversion:     {r.breakeven_rate:.0%}")
    print(f"  EP  go {fmt(r.ep_go)} | punt {fmt(r.ep_punt)} | FG {fmt(r.ep_fg)}")
    print(f"  Recommendation:            {r.recommendation.upper()}"
          + ("  (can't get the ball back: only go/FG-to-tie-or-win count)" if r.desperation else "")
          + ("  (last play of the half: only points on this play count)" if r.last_play else ""))
    return r

In [12]:
# 3. Example situations
# =====================================================================
evaluate(qtr=4, clock="0:30", yardline_100=1, ydstogo=1, score_diff=-10, timeouts=3)
evaluate(qtr=4, clock="0:30", yardline_100=20, ydstogo=3, score_diff=-2, timeouts=1)
evaluate(qtr=1, clock="10:00", yardline_100=60, ydstogo=1, score_diff=0)
evaluate(qtr=3, clock="7:00", yardline_100=35, ydstogo=4, score_diff=0)
evaluate(qtr=2, clock="5:00", yardline_100=75, ydstogo=6, score_diff=7)
evaluate(qtr=2, clock="0:01", yardline_100=49, ydstogo=1, score_diff=-10)
 



Q4 0:30 | 4th & 1 at opp 1 | down 10 | 3 TO
  P(convert):                69%
  Points added if you make:  +6.69  (vs best kick/punt)
  Points lost if you miss:   +0.00
  Break-even conversion:     0%
  EP  go +4.95 | punt n/a | FG +0.31
  Recommendation:            GO  (can't get the ball back: only go/FG-to-tie-or-win count)

Q4 0:30 | 4th & 3 at opp 20 | down 2 | 1 TO
  P(convert):                50%
  Points added if you make:  +2.64  (vs best kick/punt)
  Points lost if you miss:   -3.30
  Break-even conversion:     56%
  EP  go +2.26 | punt n/a | FG +2.61
  Recommendation:            FIELD_GOAL  (can't get the ball back: only go/FG-to-tie-or-win count)

Q1 10:00 | 4th & 1 at own 40 | tied | 3 TO
  P(convert):                69%
  Points added if you make:  +2.82  (vs best kick/punt)
  Points lost if you miss:   -2.64
  Break-even conversion:     48%
  EP  go +0.61 | punt -0.54 | FG n/a
  Recommendation:            GO

Q3 7:00 | 4th & 4 at opp 35 | tied | 3 TO
  P(convert):       

p_convert            0.693425
ep_if_convert        0.037691
ep_if_fail                0.0
ep_go                0.026136
ep_punt                   0.0
ep_fg                     NaN
best_alt_ep               0.0
points_if_convert    0.037691
points_if_fail            0.0
breakeven_rate            0.0
go_advantage         0.026136
recommendation             go
desperation             False
last_play                True
Name: 0, dtype: object

In [13]:
# 4. Grade every real 4th-down decision in EVAL_SEASON
season = pbp_eval[
    (pbp_eval["season"] == EVAL_SEASON)
    & (pbp_eval["down"] == 4)
    & pbp_eval["play_type"].isin(["run", "pass", "punt", "field_goal"])
    & clean_eval
    & pbp_eval["yardline_100"].notna()
    & pbp_eval["ydstogo"].notna()
].copy()
season["decision"] = season["play_type"].map(
    {"run": "go", "pass": "go", "punt": "punt", "field_goal": "field_goal"}
)
 
ev = season.join(evaluate_frame(season))
col_for = {"go": "ep_go", "punt": "ep_punt", "field_goal": "ep_fg"}
ev["ep_chosen"] = np.select(
    [ev["decision"] == d for d in col_for], [ev[c] for c in col_for.values()], np.nan
)
ev["ep_best"] = ev[["ep_go", "ep_punt", "ep_fg"]].max(axis=1)
ev["ep_lost"] = ev["ep_best"] - ev["ep_chosen"]  # 0 = made the model's call
ev["disagree"] = ev["decision"] != ev["recommendation"]
 
print(f"\n=== {EVAL_SEASON}: how often teams matched the model ===")
print(pd.crosstab(ev["decision"], ev["recommendation"], margins=True,
                  rownames=["actual"], colnames=["model"]))
 
team = (
    ev.groupby("posteam")
    .agg(
        fourth_downs=("decision", "size"),
        went_for_it=("decision", lambda s: (s == "go").sum()),
        model_said_go=("recommendation", lambda s: (s == "go").sum()),
        disagreed=("disagree", "sum"),
        ep_lost=("ep_lost", "sum"),
    )
    .sort_values("ep_lost", ascending=False)
    .round(2)
)
print(f"\n=== {EVAL_SEASON}: expected points lost to 4th-down decisions, by team ===")
print(team)
 
print(f"\n=== {EVAL_SEASON}: 10 costliest decisions ===")
worst = ev.nlargest(10, "ep_lost")[[
    "week", "posteam", "qtr", "game_seconds_remaining", "score_differential",
    "yardline_100", "ydstogo", "decision", "recommendation", "ep_lost", "desc",
]].round(2)
print(worst.to_string(index=False))


=== 2025: how often teams matched the model ===
model       field_goal   go  punt   All
actual                                 
field_goal         991   29     4  1024
go                 260  550   121   931
punt                71  146  1825  2042
All               1322  725  1950  3997

=== 2025: expected points lost to 4th-down decisions, by team ===
         fourth_downs  went_for_it  model_said_go  disagreed  ep_lost
posteam                                                              
CLE               147           29             25         26    29.73
TEN               145           35             28         27    27.65
NYG               124           39             24         32    27.50
HOU               153           21             26         23    26.47
DEN               140           22             27         17    24.78
NYJ               139           41             24         29    23.91
LAC               128           23             20         19    23.37
MIN           

In [14]:
# 5. Grade every real 4th-down decision in 2024 season for disappointment modifier (d) estimate
# filter for kneels/spikes, edited to apply to 2024 evaluation data
clean_eval = (pbp["qb_kneel"].fillna(0) == 0) & (pbp["qb_spike"].fillna(0) == 0)

season_24 = pbp[
    (pbp["season"] == 2024)
    & (pbp["down"] == 4)
    & pbp["play_type"].isin(["run", "pass", "punt", "field_goal"])
    & clean_eval
    & pbp["yardline_100"].notna()
    & pbp["ydstogo"].notna()
].copy()
season_24["decision"] = season_24["play_type"].map(
    {"run": "go", "pass": "go", "punt": "punt", "field_goal": "field_goal"}
)


In [15]:
# test values of d sensitivity modifier to test against 2024 4th down decisions
for d in [0, .1, .2, .3, .4, .5, .6 , .7, .8, .9, 1]:
    dis_adj_test = evaluate_frame(season_24, d = d)
    results = (dis_adj_test["recommendation"] == season_24["decision"] ).mean() # take ratio of matching results with coach decisions
    print(d, results)

# d = 0.6 is the sensitivity modifier that provides the closest results to 2024 actual coaching decisions
print("Most accurate D value:", 0.6, "Percentage Closeness: ", 84.26, "%")

0 0.7265449086815111
0.1 0.7498123592694521
0.2 0.7900925694270703
0.3 0.8073555166374781
0.4 0.8106079559669752
0.5 0.8213660245183888
0.6 0.8426319739804854
0.7 0.8373780335251438
0.8 0.8326244683512635
0.9 0.8381285964473355
1 0.8416312234175631
Most accurate D value: 0.6 Percentage Closeness:  84.26 %


In [16]:
# compare original model results with disappointment-adjusted model at optimal d=0.6 on 2025 NFL data
original_model = evaluate_frame(season, d = 0)
dis_adj_model = evaluate_frame(season, d = 0.6)

og_results = (original_model["recommendation"] == season["decision"] ).mean() # ratio of matching results with coach decisions
dis_adj_results = (dis_adj_model["recommendation"] == season["decision"] ).mean()

# comparison results tested on 2025 evaluation season
print(f"\n=== {EVAL_SEASON}: Comparison of Models tested against 2025 4th Down Decisions ===")
print("Original Model:", round(og_results * 100, 2), "%")
print("Disappointment-Adjusted (d modifier = 0.6):", round(dis_adj_results * 100, 2), "%")


=== 2025: Comparison of Models tested against 2025 4th Down Decisions ===
Original Model: 72.68 %
Disappointment-Adjusted (d modifier = 0.6): 84.21 %
